# Final project

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tbloesch/deepLearning/blob/main/notebooks/final_project.ipynb)

The code lives in `src/dlproject/`. This notebook only imports and runs it.
Edit the code in VS Code, push to GitHub, then re-run the setup cell in Colab to pull the changes
(and *Runtime → Restart session* so the new code is loaded).

**Contents**
1. Setup
2. Data
3. Data sanity checks
4. Model
5. Training
6. Evaluation

## 1. Setup

In [ ]:
# Setup: works both in Colab and locally in VS Code
import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/tbloesch/deepLearning.git'

if 'google.colab' in sys.modules:
    repo = Path('/content/deepLearning')
    if repo.exists():
        !git -C {repo} pull -q
    else:
        !git clone -q {REPO_URL} {repo}
    !pip install -q gdown torchinfo torchmetrics
else:
    repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

os.chdir(repo)                      # data/ is downloaded into the repo root
sys.path.insert(0, str(repo / 'src'))

# Auto-reload edited modules (Colab's IPython autoreload is broken on Python 3.12+, so skip it there;
# in Colab, after pulling new code use Runtime -> Restart session and re-run the cells)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except ModuleNotFoundError:
    pass

In [ ]:
# Switches for slow steps
RUN_SLOW_CHECKS = False   # duplicate search (runs ResNet-18 over every image)
TRAIN_MODEL     = True    # False = load the saved checkpoint instead of training

In [ ]:
from dlproject.config import get_device
from dlproject.data import download_data, get_datasets, get_loader, describe_dataset, describe_batch
from dlproject import checks

device = get_device()
data_dir = download_data()

## 2. Data

In [ ]:
train_dataset, val_dataset = get_datasets(data_dir)
train_loader = get_loader(train_dataset)

describe_dataset(train_dataset)
print()
describe_batch(train_loader)

## 3. Data sanity checks

In [ ]:
checks.visualise_samples(train_dataset, train_dataset.classes)

In [ ]:
checks.check_class_distribution(train_dataset, train_dataset.classes, 'training set')

In [ ]:
checks.check_image_sizes(data_dir / 'train')

In [ ]:
checks.check_tensor_stats(train_loader)

In [ ]:
checks.check_split(train_dataset, val_dataset, train_dataset.classes)

In [ ]:
corrupt = checks.find_corrupt_images(data_dir)

In [ ]:
# In this example dataset the duplicates are intentionally identical images with the same filename.
# In real datasets duplicates may be similar but not identical, so a high threshold is used.
if RUN_SLOW_CHECKS:
    possible_duplicates = checks.find_train_val_duplicates(train_dataset, val_dataset, device=device)

## 4. Model

Define the model in `src/dlproject/models.py`, then import and build it here.

In [ ]:
# from dlproject.models import build_model
# model = build_model(num_classes=len(train_dataset.classes)).to(device)

## 5. Training

Put the training loop in `src/dlproject/train.py`.
Checkpoints are saved to Google Drive in Colab so they survive a runtime restart.

In [ ]:
import torch

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/deepLearning/checkpoints')
else:
    CHECKPOINT_DIR = repo / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / 'model.pt'

val_loader = get_loader(val_dataset, shuffle=False)

# if TRAIN_MODEL:
#     from dlproject.train import train
#     history = train(model, train_loader, val_loader, device=device, epochs=10)
#     torch.save(model.state_dict(), checkpoint_path)
# else:
#     model.load_state_dict(torch.load(checkpoint_path, map_location=device))

## 6. Evaluation

Accuracy, confusion matrix, example predictions, ...

In [ ]:
# from dlproject.evaluate import evaluate
# evaluate(model, val_loader, class_names=train_dataset.classes, device=device)